# IFAC Figure 4 — Colab replot from archived results

This notebook **does not rerun the Monte Carlo experiment**. It redraws the exact released 5-point means and pointwise 95% percentile-bootstrap bands from embedded machine-readable data. Use `figure4/recompute_figure4.py` in the source repository to regenerate the underlying 1,200-trajectory simulations.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, io, zipfile, base64
DATA_ZIP_BASE64 = "UEsDBBQAAAAIAAdYY10zOQ3BwhUAABRNAAAdAAAAZmlndXJlNF9maXZlcG9pbnRfU0VfQ0k5NS5jc3bFnE1zZMdxRff+LR0d9f2xNe2FFvbG3iNGQ0iEgxyMZyYsU7/e51Q9UOgGqNHCaFAcDfj69WtUVubNezOz+kM8fUin/z59OP35y8OPdw+ffrz/39Pnx58fPv56+np//+Pp30//efrl/sOnu4+PX7/x048PTz9//fDL55/v7/7jX3776V9Pf3x8/Pb125cPn+9++MOsdz8//uX60k8Pf/7p9MMf7n65//bT44+nh0/f7r98/nL/7cO3h8dP/xTP6ZRP4TxTuPinnHwlnB6/fPj48/0phdRSaOEUue+U+RPOIbaUaqll9pxHaqGviyXUEeYYM5QSZ69eDK30PluqvaeS0shjXQ1x9MCTexxjVF5cD6i8d846WoqRh5f9US3k3OuMk0/MMZ0+33/5eP/p2wNm+NuCH/90x9//df/x2+OXX++04te7fy4h3H1+ZN1/efh6f/rTw6eHb/d3Pz1+efjr46e7f/vh7vHTz79+1w5ft70+fPn1FVvEHs6pTW0wYy+RXy+cW5mh8biZehujp37KvZdz7FxkDT3kME4xjPOsMYYcZwmTpZ1iO/fZZyptVGyVQzsVrsVRQ8211jJKvPXqP/50/8dXfSClPnpNcYbQx/aAyEY2L/Zc0mC7vTbbiC6aldbSx3aK6sWeBpcxT4jTB8ZWSsc0PbHh+JDX+DvOETFIywmL3nr5j5+/Pfzy8FeC8zUb8HvxS6Vc2Sb2PNV1jX2rJZfCzR2vcMF5xD5ybjhzizONHQSYBEfPLedYQxl5v73OwnvjCJhwzua1Hoi02Osy0Og3t8K3vzzefbn/eP/wP79jCAKeCEi98yfNvteXx0w5DWAh91ZGnSuc+ygVn6gEAd4/tz/wY8WYPddZZyYa8npoxjtyLykCFCksjwigjoGWY4u8xkfe0hjs8sU/p3gFk/kSJie/I+A2R2sJSBwb0Cob2xI4h21K3b4wwygjtOUOYEBp2y5JoCWqQBICrrRlrjhaxl1inzU1HGVfnI0QicAGluRDb+okr9jlJWxe2gavYHmjtzYmSSRslxHu2WuSQphEwlqZoKIj5Zgrf7e4g6e1GkFNUobOtYAXR+N/1dvwDlY1txsBKRlnIfKIoT7e2TLPIfXSJiOzeBCQBYGNBjoX9Qk8Hm8ppREQXusTf6l8AsHScks7iswYJMqOnSJ4EvedxCGA0kTljG3W53QgNfSFuI0/7+0rL1D20iytGBK4QSdySlyeIinIwGODXEA78lpsSvgEoDNB5EEs5B1ErZpO2H6irHVuWQ/w/VAOEguRGMO+SHIOZnFewMVye2fDvAa8l7ZJLrjkACBEEkxbuEFWBjFMs8AybrTZlSlo9MJbBsSj9m2cTK4hOZeEhTJWXZehIPgG0BJI1BMA3h/Vx+yJ3I03mbluZ5x0jpeX4ildIW+8NAt+LQ3BRXSTRUVCkozqG+BxZRFlX5yYgOTdK6x1HKGEf5F9asywHIEnLlcCVvAqAgxMqk/BlOoUqmYSpSI/vLNZXgLvpWky2ACbDiAp7BQEPiInuoQeC8m2jLajgTwNm8cAukE6WAuAC2uHk0vi5k5JcpWSMGyX19dF6AI43ASgSQYEa9p7O8xz4L20SSM5SK9Jyuwr5G3lDQCzs/TK/YbZomcABQsacjh2e0cWJiVdZbN8nphkJ+8q0eFpvCPUFDbAdEBJjoShEx95S/byegxdI++lXZBiJBIIOgmWV3cYFXG4l2kGwUA7YqAs5B18A9rfaxwbehNeEYHjNpqvbPFXq3wYMYD/DMXO3gGcUS8JvJS4+s6GeQ15ryCmIfzaNAcRHGzmgbygKEEDEnRReQcXPpFhMMAmnlbiU1pKSuIwRJ9EiGw0wWa4BmgORSKk1kX4M++eWBktAMW7pXHqZVoa3HiJvOUSXnJD7C7ay894zV4VKl//mTL2fgQIyrGjHRdJNr1veIESq/9Hl8f0TY+R/ZFwIr0FeHI/yKKlBZgMUgs/JPTe2SovgbdcMd6oUgZjunpwZ1rQYEBYoafghwTwiJCwpBOWMrLmtgxR1wdSa4i+EODNeAuwTYDiw/jg2ISXlJdWJWbB8Xsb5jnuXpiEZAPzyFYC2jSeFjkdyDoFf6lEQlkkOFsxIK+kYZJKO1S4M60yCRDT9AHjKsJPdApCiuBMiy4DKlUr8m5cDa3+3uFzDbqXRqlRjvqUIjZhj5EwQRgP/aIsrBFIYKkoJQg+BHlXy5QGvetNhEVb5ZaooqjJUhNwPRf84H61Ras0IBXyqLyzSV6D26voqdhkWmXDMNYZNtjg8jj+1M1H2X4B8yfO4G58DC/XdIijDrKCwnlCZJ8iRZ00YT41kOTKVgaANBEFhBupIO8NwyefyhW4pst8DC3P5FKYVmb32qZZACNkI5QNMIuQsafRmiHqqE2AeKecCvlH4gA4IAb8r2/Dwucy+BoJuVlWGiNfq0QxYhOo0g2ZmyZ4iaTPzJB1/ZarLDPj7RghWnwma3AxuaX8vuihMwYhNrAMkAJTi4vQoI+tuuL8p1jOYVaLbNHCM7F0qo1LFrDItCPcNKe48OdI+WzJSUEDiQbYIO4V0sETalplDlLBRMviE6eYzz10i8Udr66r3IS4xbMtq4GBBIbGsoBWC+iYyDjoJJ6vNBx4O7BBysg33u8XgPhs7fFs7RcSTeDCt+qq6ZSIrzeknFU1vJfNZVmsZ3HHpHRbaQO+UQZJgNxgeoTAnc21DaqOa5CFcR7wCDMgT7rUs/Ipt138a9B3EfXTug4sWoyqyPq54hualN2vDh/i9z5V+wQ1BgIfARJWUTAWc+AQ42y4jNWcqKjcxh1DDUsutcaL9FWWiJfQ0vpmBqiXFY72SqsphqtWE0nfYimJPVrCWRwhZgugQDzEgIXs0jKafSBWZh7aa5SDNgUL58o6g4JFH1XYYp6EKrj2Ujb5XAgIX4LRQyhiertK4e8Y4hr4royRhCThrFqdmCvdsWPc0WwccQng4lGWmNGroIU9iJwOXl55U8qgv7TQi5AhIEW6mMx6y2DwgpKt/TQsXt6yyvM7NvgbBl6vHuRroLNhPo375eGZBJV3KiDhr2vISglUaXEVxXfZAqjgQ8A3VmXRfN05m50W6WfDc6JZMuEEsIkm/Y640tv1G38vGC7B8MoI0R7rFlmA4VghTcYjiRPfJra5K8ToyoI+J99BCPCWdGjOLLcBSrKaKy/Cs+pBpJKSiwIrr0skkgYRBzNAhvZ29PB3jPASFK9xoQR+V9R0gM7NegjlrFwCFzMIT1zvFnS0L2IayGwoHO9orRT2myckU0iPO6cGQgN3SiTU0s24myTy+Kr6bpCJ/oYFvleM8Z2OE5z+qjFP/i/WfEvTUbYoINcDfygigb+2ozFvC4HVFADAZvtTY35aqyBamvXgo9qOCbu3QqfgS7CpdRGBnnstPLQqL26Klv9Ax+naNglEI2CsFaxC5zwKn6JKAlRIMGM3TaLECG2RLYwTFkfw4FUdUUVMoM8Bo105rboKltlye0uIpHHBo2RlYlprf1/TPMfUq94KtsjWaC2DQx92hY8dDn2QZgNMMW71ALNKCNPgFEPrpeWjKGFYJKVohpzu/gPsPJFZbR7Ay9PO2QhYPWasvoK9z/cOo2ucvepPpphVglBJkGTsol2xah4tZyPEWfST5ma5q0QuoYxHIFlnD2sOBiyuu90ClEzAeELUlO1HR9vGLsaODaOxBe/tLK9h71UkRUcT5mJJ/Aa7WxDnEotEFvtvsW7XaEy4AEeHX/dc6tGSs5e0DKnIJP7Gfix6xnmSTK42++wARa7gd10ajF/eEH2/23Wya3jZ7y9p6Cw5AQtxV7Yj+qvbayHNsr680XeQoXCOWEe2cpXik13sMHQb+VOk3kCLmCuOT0FLpvizPwspOEM2yLoK8J3t8hJ9r8rlVm+ys0/OxuxwgplzESZCGiHIjoac6bo2kAXcLGEe4dRhOMp9eC2hQkpe95LG0EBRVY9l8kF9RBpnR9AN3H1D8vKdttO1TYoNSlWJOsYq5aIgi78EMHU4V7XBtzsdAtNJhlN/ykgmL1I3SV+L1Q1HYhT/jW1heNZVN6ID2XUqonDMWW+Yq/+RvtO1YWzJ927hArkDLG7DpFkk8IAJCgVUXVnFMQmLHd5oc2VnpWkpTMULAebv/f42m/OIDg4Bswf4WhbszTkcLTxuCL7/YN/pRSA5oED+SKukXbYZgEiW6XBVt7ZZthmIA6ROsxkOckBeD5BpTk8RHESJs2rxwJM1y9ksp2O5zY7Qwg5bWTjVgW4KMt9pPMVwWSBOkjpYDLwjssO7QOz4IdJfZZOsg25BsDFWwPCGp5Z/dgTCalJefc12yChS0podscu7wcjWTAd3tPa0p/DOVnkJveW6UwkirOnD8VvpG8KLRxBfjjTU2Y66ClYcq0DkcFo4KijZRi3KsFbr4rt7l0AeByQQTNXGXHrivSBPs3YwSgo3hZm/23m6tglxgfydzoPATeIeOHSgDLBw5AMwKXvSrDpVxfLLHJoxHONnwqh0paqr5oZeUMhqs/XKWp86Dw08b0I2ngYc15sSmO93n64NA3OTndm3h3D03Q9A/aKg2ftpBt8tf1KMeJudSbUItUchGiFoJsd9SNdHSiLNW1RKw0FgfGg5kHNqIDq2sQeOuHxnu7yGvFdxBAcrkNChvpalbDBwnAHVDbmxhnbE0bQdYWHOuvpRcAjITwRCcuDbcbwdMTagrL53oWgPzljnc5aGuMXW6Zby8boDpSK8tAG5s4ANyONNwTL8K0Wb8iSLY+KB/7RqItuNhEl9KrkAGpUkZUHCSdUNJGQg5xqyn2crblsgIRZ5+Bp8JjBva4GXiHphBbiGdUNovi21PabqGJ71xIZwZNEr/QAboufirqBleZoGsV7gv5DZPHd/m2CqzkirO/dMFXK6YRdMOB25STesyl+2oq4Xj0xz1mmuAbLt8A5IsfWw0aAsWckE2lGtFCDckiCxHWMsAWDngXTUi1YCBMhKa0YVmFyz47nCV2tMhAuB1d6wPf16AFxD5IUBysTViwNt3eMLC80ABiilUD8cbAJQoj125xswFFfinp6DYnVTbl/1tbHGpMgMZU21aFnrSVyrFkhQB06D4zC3LMC+3pO6sgE/5upMUgS/ytF4707tAFxdejQAd3dSekEWSfYeR9v634qz49jwX0vRu4/B3058z+gszOafOMWwIYUMtApX3gAH66mcjTb041VG8LWr1lS9bk0Rm9aFixXTY0bUtOlwKWwT6ZE3xoXKy7abUlzYno8yiPOTpBNCnS2OZQ8sO9QBvqbFLkI/itikYDksuYP8OscbOMT3TPEMEZ4bopzxUIs+0D5EvP6bz6S1YLHQ1C74cVdRqZs9moX43biAgzbwH9sgZxz1KWccB0IViCtJN7ysEkjJwe+O5YZDpe/gBVeAcLl+lIVlnLAmiy1s5vOEB0OEBv/vIJLP6U5pQR6sVuQN+zYveU4RNpuNC3+PyOZCqaShxFH3Up5rjsHDHqBEe4NM8D0DvAIIl8FQ/AU9z0YgEO2rDJ4Hmd+jCgG5WdvGSYcrUCEs3ZLeJo2ERegDQo3ORH+WBYrVkUgpk5p28fNiNUBpjw1B27eYzv99O6QzGfiq+VKvmi/Do2CwtuIYcdz9AZbulKjOUK1y7cqNmd3xGlk14NB+a76wfHRGXmMqbVNEgqOj1gs4YRf7KP+tsCLpDKuD7S3YwXdt8YIlXdpDgjSkALxHYFge3xWXmKGtU3Me0Yi2oZuzetOh4rIbUbahocXJQ2IOV6+upMNb1jLgFGsEX4fCMN3+dUS4hvEW9Zvv2uE5MF5YoKdmNnCKqjrluUZXPEJBrgtSgtLXzCMubrmSJWTTxnGYpVZyxQr9bGNACzRDJi0zkE3SAlG8A5gBfLHiiPEtZrW+HxXX+HhhBtbhuS2Z8cQcqwHr0IXji1kEREEsvFjnSXGFEp11OqTUOn7g/HB3wGUtuTrlFGRVjjSm1VsRQ1Mx4axDle9hhddA8vrQG0S3DRYIFWDRR8F3eOSxpOS+xvx07NnjB9EDWo53HS1IT/k4Gl2s9mKO3fUWU8ETD0mrJNZTi4dmoRBgcVZe3xQg8nWjpF4fz1FIsuYenMc8zpCoJR1AKqOvctVWS+DkGk0MNpOeZnqacVFwKyAGtXnI5iQozq4HtbRnI+TT0WNAagz4Sb1p0sivNUaubQExkDqT0+wiPy1FpogvIJPrPlXOSqBAgGJJ60jf0YANfdEOBBMErD91VtbhBGINE6kzD0SZjrU4Lup48FtMN/19SzzHyQsbKKdXLM9oPWlFgHDu0Mc+txmWfFIk6eEO+7anM1kqL3l084hB3aPRzSKkA1F5DQ0tQlFX7ao76eO05E1ZZH6l5XFpAg992KCxfp08LrNU9TBz4PQeOgtLHYAEdkdX/TKmnReXnnIiyrHWOOcqOFmqTBaVJEs9bkKmrICPjLqY+rh5LLyGkFedn2FeWBX3uk6orov2gPm1yfIen951VUsFTddmVZ7VPE6nERDLOYIjs8dUS26OdPG4of6uGx+D5/W8FX4uV7ulKc5wl6tuRr2qNc62x9CHx+fHeJo1STbvTBXdedhdgINKOcbkgKSDXQdCOgpckJrBqYSjXUoq9fhQmOvAzWHH6Ol7bnRi3vLtO1jiJUZeWqOa2gHxuQa7jjaMBzqggp6CduBk93YalKrm7neFsNyZjn7g8NCqitS5xrpH5JyOd3iSDxqYeBewm/ORyfLTtM7Vbxsi2xrPcfKZHdJ5HXVY5/4bG75grXkCJJoWUiTFwUPSPNvHREDkMfc5ibGS5yC24rKfc9AeEQcJh5YjgBo6G1fIw9G/6GRxuy1CHiFxDZLP1h/Pcp/99RgEdl78ULoQPc4Q8IXhoap6jiY4e+XkQ4+6r0rSGpr1nMgGFq0CoDhJWsydDqM4Gu+xteKj9uG+dzDBaxh5GQ3wXtbSPVvZQ2ub55AEo/sMibCOtutnw4Pt3a/e6ZYbdzBUMMFv33B6ZxwHnnmKVdmJ6PCLBfYYHEKtO1zZLGpCwG7qEF6/akXUdDU9ET24UvwWCL8z4hhzRk3Dme1Uhqfvm+k2J4Z8wl7VPFpVSWplbxgjdnjkE5fmUVG0sOayp7Z6V6n20q3KkXdub4fnkPD8LEz3dIQiCgP4RULl1D0DQuinbsNpQqn89iFuGw60W2XiRc/COOHnkLPncldTLJ27h4qQq/jQ8K5GkAATzmfJzW+srg8PuEaE52dDuvVr+OIaDkv2GJpfDWGHaQpwnptgrRO52eFWKa7J2E6oSas8xb2+lsjJiJjxE4trrNh54oisK+e5BFlxLgLyeVNRuZf/Gho8s8AiyaBbtwdnHbXwvmE/1STvaH+vbm3l31U1QiGkwhPIG7Gd15Gf4PeA9HEaBTvZnqur4g6TTn49BOIWLiLnTv//M5v/B1BLAQIUAxQAAAAIAAdYY10zOQ3BwhUAABRNAAAdAAAAAAAAAAAAAACkgQAAAABmaWd1cmU0X2ZpdmVwb2ludF9TRV9DSTk1LmNzdlBLBQYAAAAAAQABAEsAAAD9FQAAAAA="
with zipfile.ZipFile(io.BytesIO(base64.b64decode(DATA_ZIP_BASE64))) as z:
    df=pd.read_csv(z.open('figure4_fivepoint_SE_CI95.csv'))
print(f"Loaded {len(df)} controller/plant records")
display(df.head())

In [ ]:
SCENARIOS=[(1.2,3.0,.92),(1.2,3.0,.96),(1.5,4.5,.98)]
POLICIES=['oracle','stationary','cheb','optimized','two_received']
LABEL={'oracle':'Known-plant oracle','stationary':'Stationary','cheb':'Chebyshev','optimized':'Weighted periodic','two_received':'Two received states'}
fig,axs=plt.subplots(1,3,figsize=(16.5,4.7),layout='constrained')
for ax,(lo,hi,q) in zip(axs,SCENARIOS):
    sub=df[np.isclose(df.a1,lo)&np.isclose(df.a2,hi)&np.isclose(df.q,q)]
    for policy in POLICIES:
        ss=sub[sub.policy==policy].sort_values('a')
        if policy=='two_received' and np.isclose(q,.98): ss=ss[ss.a<hi-1e-8]
        if ss.empty:continue
        line,=ax.plot(ss.a,ss.mean_cost,marker='o',lw=1.3,markersize=3.5,label=LABEL[policy])
        ax.fill_between(ss.a,ss.bootstrap_CI95_low,ss.bootstrap_CI95_high,color=line.get_color(),alpha=.15,lw=0)
    ax.set(title=rf'$a\in[{lo:g},{hi:g}],\ q={q:.2f}$',xlabel=r'Plant pole $a$',yscale='log',ylabel=r'$J_T$ (finite horizon, $T=300$)')
    ax.set_xticks(np.linspace(lo,hi,5));ax.grid(alpha=.18);ax.legend(fontsize=6.9,loc='upper center')
    if np.isclose(q,.98):
        ax.set_ylim(.008,120);ax.scatter([hi],[100],marker='^',s=36,color='tab:purple',clip_on=False)
fig.suptitle('Paired finite-horizon costs; shaded pointwise 95% bootstrap CIs',fontsize=11)
fig.savefig('fig03_finite_horizon_mc.png',dpi=220,bbox_inches='tight')
plt.show()